In [2]:
# journal_07_jetson_eval.ipynb -- one cell. Runs on the laptop, no GPU needed.
# Reads the Jetson's results folder (copied to journal\jetson_results) and produces:
#   1. mAP of every TensorRT engine (FP32 / FP16 / INT8) on FLIR night + full, computed with exactly the
#      same matching and AP code as every other number in the paper, next to the laptop PyTorch number
#   2. the latency / FPS / power tables from bench_15W.json and bench_MAXN.json (trtexec + tegrastats)
#   3. the end-to-end Python pipeline timings (decode + letterbox, inference, NMS) from the accuracy runs
# The Jetson pipeline timings in table 3 were measured in 15 W mode.
# Everything is saved to journal\results\jetson_summary.json.
import json

import numpy as np
import torch
from torch.utils.data import DataLoader

import lb_fusion as L

RES = L.JOURNAL / "jetson_results"
ds = L.DATASETS["flir"]
NAMES = ds["names"]
_, _, EVAL_SETS = L.get_splits("flir", smoke=False)          # night (316) and full (1013), same as every run
MODELS = {"yolo_A": "A (RGB only)", "yolo_B_s1": "B s1", "yolo_B_robust_s1": "B_robust s1",
          "yolo_C_s1": "C s1", "yolo_T": "Thermal only"}
PRECS = ["fp32", "fp16", "int8"]


# ---------------------------------------------------------------- ground truth (letterboxed 640 px, as in evaluate_map)
def load_gt(ids):
    loader = DataLoader(L.PairedLetterboxDataset(ds, "val", ids), batch_size=16, shuffle=False,
                        collate_fn=L.collate_meta_fn, num_workers=4)
    gt = {}
    for _, _, bd, img_ids, _ in loader:
        for i, img_id in enumerate(img_ids):
            m = bd["batch_idx"] == i
            gt[img_id] = (bd["cls"][m].clone(), L.xywh2xyxy(bd["bboxes"][m]) * L.IMG_SIZE)
    return gt


# ---------------------------------------------------------------- mAP from saved detections (mirrors evaluate_map)
def load_dets(path):
    z = np.load(path)
    ids = [str(i) for i in z["ids"]]
    idx = z["img_index"]
    starts = np.searchsorted(idx, np.arange(len(ids) + 1))   # detections are stored image by image, in order
    dets = {k: z["dets"][starts[n]:starts[n + 1]] for n, k in enumerate(ids)}
    times = {k: z[k] for k in ("t_pre_ms", "t_infer_ms", "t_nms_ms")}
    return dets, times


def map_from_dets(dets, gt, ids):
    all_tp, all_conf, all_pred_cls, all_target_cls = [], [], [], []
    for img_id in ids:
        gt_cls, gt_box = gt[img_id]
        all_target_cls.append(gt_cls.numpy())
        d = dets.get(img_id)
        if d is None or len(d) == 0:
            continue
        d = torch.from_numpy(np.ascontiguousarray(d, dtype=np.float32))
        pb, pc, pcls = d[:, :4], d[:, 4], d[:, 5]
        if gt_cls.shape[0] == 0:
            tp = np.zeros((pb.shape[0], L.IOUV.shape[0]), dtype=bool)
        else:
            tp = L._match_predictions(pcls, gt_cls, L.box_iou(gt_box, pb), L.IOUV).numpy()
        all_tp.append(tp); all_conf.append(pc.numpy()); all_pred_cls.append(pcls.numpy())
    if not all_tp:
        return {"precision": 0.0, "recall": 0.0, "map50": 0.0, "map50_95": 0.0, "per_class_ap50": {}}
    tp, fp, p, r, f1, ap, uc = L.ap_per_class(np.concatenate(all_tp), np.concatenate(all_conf),
                                              np.concatenate(all_pred_cls), np.concatenate(all_target_cls),
                                              names=NAMES)[:7]
    return {"precision": float(p.mean()), "recall": float(r.mean()),
            "map50": float(ap[:, 0].mean()), "map50_95": float(ap.mean()),
            "per_class_ap50": {NAMES[int(c)]: float(ap[i, 0]) for i, c in enumerate(uc)}}


# ---------------------------------------------------------------- laptop (PyTorch) reference numbers
def laptop_reference():
    ref = {}
    rob_path = L.JOURNAL / "results" / "robustness_flir.json"
    rob = json.loads(rob_path.read_text()) if rob_path.exists() else {}
    rob_names = {"yolo_A": "A (RGB only = detached adapter)", "yolo_B_s1": "B s1",
                 "yolo_B_robust_s1": "B_robust s1", "yolo_T": "Thermal only"}
    for split in EVAL_SETS:
        for m, rn in rob_names.items():
            r = rob.get(f"{split}|{rn}|clean")
            if r:
                ref[(m, split)] = r
        c_path = L.JOURNAL / "results" / "lb_yolo_flir_C_s1_eval.json"
        if c_path.exists():
            r = json.loads(c_path.read_text())["eval"].get(split)
            if r:
                ref[("yolo_C_s1", split)] = r
    return ref


print("loading ground truth (1013 FLIR val images)...")
GT = load_gt(EVAL_SETS["full"])
REF = laptop_reference()

accuracy, pipeline = {}, {}
for m in MODELS:
    for prec in PRECS:
        path = RES / f"dets_{m}_{prec}.npz"
        if not path.exists():
            print(f"(missing {path.name})")
            continue
        dets, times = load_dets(path)
        missing = set(EVAL_SETS["full"]) - set(dets)
        if missing:
            raise RuntimeError(f"{path.name}: {len(missing)} validation images have no entry")
        for split, ids in EVAL_SETS.items():
            accuracy[f"{split}|{m}|{prec}"] = map_from_dets(dets, GT, ids)
        t_pre, t_inf, t_nms = (times[k] for k in ("t_pre_ms", "t_infer_ms", "t_nms_ms"))
        total = t_pre + t_inf + t_nms
        pipeline[f"{m}|{prec}"] = {"median_ms": {"preprocess": float(np.median(t_pre)), "inference": float(np.median(t_inf)),
                                                 "nms": float(np.median(t_nms)), "total": float(np.median(total))},
                                   "fps_end_to_end": float(1000 / np.median(total))}

# ---------------------------------------------------------------- table 1: accuracy
for split in EVAL_SETS:
    print(f"\nTABLE 1 ({split}): mAP50 / mAP50-95 of the TensorRT engines on the Jetson")
    print(f"{'model':<16}{'laptop PyTorch':>18}" + "".join(f"{p.upper():>18}" for p in PRECS) + f"{'FP32 - laptop':>16}")
    for m, label in MODELS.items():
        ref = REF.get((m, split))
        row = f"{label:<16}" + (f"{ref['map50']:>10.3f} / {ref['map50_95']:.3f}" if ref else f"{'-':>18}")
        for prec in PRECS:
            r = accuracy.get(f"{split}|{m}|{prec}")
            row += f"{r['map50']:>10.3f} / {r['map50_95']:.3f}" if r else f"{'-':>18}"
        r32 = accuracy.get(f"{split}|{m}|fp32")
        row += f"{r32['map50'] - ref['map50']:>+16.4f}" if (ref and r32) else f"{'':>16}"
        print(row)

# sanity check: FP32 engine vs laptop PyTorch on identical inputs and identical evaluation code
diffs = [abs(accuracy[f"{s}|{m}|fp32"]["map50"] - REF[(m, s)]["map50"])
         for (m, s) in REF if f"{s}|{m}|fp32" in accuracy]
if diffs:
    worst = max(diffs)
    print(f"\nFP32 engine vs laptop, largest mAP50 difference: {worst:.4f} -> "
          + ("OK, the Jetson pipeline reproduces the laptop numbers" if worst < 0.005 else
             "LARGER THAN EXPECTED -- send me this output before using the Jetson numbers"))

# ---------------------------------------------------------------- table 2: speed + power (trtexec, batch 1), per power mode
BENCH_FILES = {"15W": "bench_15W.json", "MAXN_SUPER": "bench_MAXN.json"}
speed, systems = {}, {}
for mode, fname in BENCH_FILES.items():
    path = RES / fname
    if not path.exists():
        print(f"\n(missing {fname} -> no {mode} table)")
        continue
    bench = json.loads(path.read_text())
    idle_w = bench["_idle_power_mW"]["VDD_IN"] / 1000
    systems[mode] = {"nvpmodel": bench["_system"]["nvpmodel"].splitlines()[0], "idle_power_mW": bench["_idle_power_mW"]}
    speed[mode] = {}
    print(f"\nTABLE 2 ({mode}): TensorRT latency and power on the Jetson Orin Nano  "
          f"[{systems[mode]['nvpmodel']}, idle board power {idle_w:.2f} W]")
    print(f"{'engine':<24}{'GPU ms (median)':>16}{'p99':>8}{'+ vs A':>14}{'FPS':>8}{'board W':>9}{'mJ/frame':>10}")
    for arch, models in (("yolo", ["A", "B_s1", "B_robust_s1", "C_s1", "T"]), ("nanodet", ["A", "B_s1", "C_s1"])):
        for prec in PRECS:
            base = bench.get(f"{arch}_A_{prec}")
            for m in models:
                name = f"{arch}_{m}_{prec}"
                r = bench.get(name)
                if not r:
                    continue
                med = r["gpu_compute_ms"]["median"]
                over = med - base["gpu_compute_ms"]["median"] if base else float("nan")
                pct = 100 * over / base["gpu_compute_ms"]["median"] if base else float("nan")
                w = r["power_mW"]["VDD_IN"] / 1000
                speed[mode][name] = {"gpu_ms_median": med, "gpu_ms_p99": r["gpu_compute_ms"]["p99"],
                                     "host_latency_ms_median": r["host_latency_ms"]["median"],
                                     "overhead_vs_A_ms": over, "overhead_vs_A_pct": pct, "fps": r["throughput_fps"],
                                     "board_W": w, "board_W_above_idle": w - idle_w,
                                     "mJ_per_frame": r["energy_per_frame_mJ"]}
                print(f"{name:<24}{med:>16.2f}{r['gpu_compute_ms']['p99']:>8.2f}{over:>+8.2f} ({pct:+3.0f}%)"
                      f"{r['throughput_fps']:>8.0f}{w:>9.2f}{r['energy_per_frame_mJ']:>10.1f}")
            print()

# ---------------------------------------------------------------- table 3: end-to-end Python pipeline
print("TABLE 3: end-to-end per-image time in the Python pipeline on the Jetson (median ms; preprocessing = "
      "JPEG decode + letterbox on the CPU, inference includes host<->GPU copies, NMS in numpy)")
print(f"{'engine':<24}{'preprocess':>12}{'inference':>11}{'NMS':>8}{'total':>8}{'FPS':>7}")
for k, v in pipeline.items():
    t = v["median_ms"]
    print(f"{k.replace('|', '_'):<24}{t['preprocess']:>12.1f}{t['inference']:>11.1f}{t['nms']:>8.1f}"
          f"{t['total']:>8.1f}{v['fps_end_to_end']:>7.1f}")

out = L.JOURNAL / "results" / "jetson_summary.json"
out.write_text(json.dumps({"power_modes": systems,
                           "accuracy": accuracy,
                           "laptop_reference": {f"{s}|{m}": r for (m, s), r in REF.items()},
                           "speed": speed, "pipeline": pipeline}, indent=2))
print(f"\nsaved: {out}")

loaded holdout split: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\splits\flir_train_holdout.json
loading ground truth (1013 FLIR val images)...

TABLE 1 (night): mAP50 / mAP50-95 of the TensorRT engines on the Jetson
model               laptop PyTorch              FP32              FP16              INT8   FP32 - laptop
A (RGB only)         0.451 / 0.186     0.451 / 0.186     0.452 / 0.186     0.438 / 0.176         +0.0006
B s1                 0.582 / 0.278     0.581 / 0.278     0.581 / 0.278     0.540 / 0.238         -0.0006
B_robust s1          0.572 / 0.257     0.572 / 0.257     0.572 / 0.257     0.498 / 0.216         +0.0000
C s1                 0.591 / 0.281     0.592 / 0.281     0.590 / 0.281     0.523 / 0.241         +0.0003
Thermal only         0.794 / 0.414     0.794 / 0.414     0.794 / 0.413     0.730 / 0.376         +0.0001

TABLE 1 (full): mAP50 / mAP50-95 of the TensorRT engines on the Jetson
model               laptop PyTorch              FP32              FP16